**Case ID:** PROM-0001

**Query:** How many scrape targets is Prometheus monitoring, and how many of them are down right now?

**Layer:** 3 (agent task) · **Server:** prometheus-main · **Fixture:** recorded · **Bug class:** logic

**Case description:** The answer must state the total number of scrape targets and how many are down, matching what Prometheus reports. The expected numbers are not typed in by hand: they come from the gold calls below, so the recording is the source of truth.

**Tools used:** health_check, list_metrics, execute_query

**Known limits:** in `live` mode the counts can change between the gold calls and the agent's calls. Use `replay` for a stable verdict.

In [ ]:
SERVER = "prometheus-main"
MODE = "replay"          # replay | record | live
CASE_ID = "PROM-0001"
SERVER_URL = None        # override servers.json, e.g. a local stand-in server
FIXTURES_DIR = None      # default: mcp-evals/fixtures
MODEL = "claude-sonnet-5-5"
K = 3
RUN_AGENT = True

# Set Up

In [ ]:
from evalkit import connect, asserts as A
from evalkit.lint import lint_tools

mcp = await connect(SERVER, mode=MODE, case=CASE_ID, url=SERVER_URL, fixtures_dir=FIXTURES_DIR)
print(f"{SERVER} · mode={MODE}")

# Initial Assertion
1. The server is healthy.
2. The tools this case uses are listed and pass spec lint with no errors.
3. Prometheus has the `up` metric.

In [ ]:
TOOLS_USED = ["health_check", "list_metrics", "execute_query"]

# 1. The server is healthy.
health = await mcp.call("health_check", {})
assert health.ok, f"Assertion 1 failed: health_check returned an error: {health.text}"

# 2. The tools this case uses are listed and pass spec lint with no errors.
tools = {t["name"]: t for t in await mcp.list_tools()}
missing = [t for t in TOOLS_USED if t not in tools]
assert not missing, f"Assertion 2 failed: not in tools/list: {missing}"
findings = lint_tools([tools[t] for t in TOOLS_USED])
for f in findings:
    print(f"  lint {f.severity}: {f.tool}: {f.rule}: {f.message}")
errors = [f for f in findings if f.severity == "error"]
assert not errors, f"Assertion 2 failed: {len(errors)} spec lint errors (listed above)"

# 3. Prometheus has the `up` metric.
metrics = await mcp.call("list_metrics", {})
assert A.mentions(metrics.text, "up"), "Assertion 3 failed: metric 'up' not listed"

print("Initial assertion passed")

# Action

**Simulated User**: How many scrape targets is Prometheus monitoring, and how many of them are down right now?

In [ ]:
QUERY = "How many scrape targets is Prometheus monitoring, and how many of them are down right now?"

r_total = await mcp.call("execute_query", {"query": "count(up)"})
r_down = await mcp.call("execute_query", {"query": "count(up == 0) or vector(0)"})
TOTAL, DOWN = int(A.scalar(r_total)), int(A.scalar(r_down))
print(f"total={TOTAL} down={DOWN}")

**Action Agent**: Prometheus is monitoring *TOTAL* scrape targets, and *DOWN* of them are down.

# Golden Answer

In [ ]:
GOLD_ANSWER = f"Prometheus is monitoring {TOTAL} scrape targets, and {DOWN} of them are down."
print(GOLD_ANSWER)

# Final Assertion
1. The answer states the total number of targets.
2. The answer states how many are down ("none" is accepted when zero are down).
3. The numbers came from a query: `execute_query` or `get_targets` was called.

In [ ]:
import re

def check(answer, calls, results):
    # 1. The answer states the total number of targets.
    assert A.mentions(answer, TOTAL), f"Assertion 1 failed: total {TOTAL} not in answer"
    # 2. The answer states how many are down.
    zero_ok = DOWN == 0 and re.search(r"\b(none|no targets)\b", answer, re.I)
    assert A.mentions(answer, DOWN) or zero_ok, f"Assertion 2 failed: down count {DOWN} not in answer"
    # 3. The numbers came from a query.
    assert A.called(calls, "execute_query") or A.called(calls, "get_targets"), "Assertion 3 failed: no query made"

check(GOLD_ANSWER, mcp.calls, [r_total, r_down])   # proves the case itself is right
print("Final assertion passed (gold path)")

# Agent Run
Runs the query with `MODEL`, `K` times, through the same MCP session, graded by the same `check`. Skipped when `RUN_AGENT` is False or no `ANTHROPIC_API_KEY` is set.

In [ ]:
import os

if not RUN_AGENT:
    print("Agent run skipped: RUN_AGENT is False")
elif not os.environ.get("ANTHROPIC_API_KEY"):
    print("Agent run skipped: ANTHROPIC_API_KEY is not set")
else:
    from evalkit import agent
    report = await agent.run(mcp, QUERY, check, model=MODEL, k=K)
    report.print_summary()

In [ ]:
await mcp.close()